In [ ]:
!pip install -U "transformers>=4.46" "trl>=0.12" "peft>=0.13" \
               "datasets>=3.0" "accelerate>=1.0" "bitsandbytes>=0.44" \
               tensorboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 18.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:
      Successfully uninstalled transformers-5.18.0
  Attempting uninstall

In [ ]:
import datasets

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

model_ID = "Qwen/Qwen2.5-7B-Instruct"

Quantization_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant= True,
    load_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype= torch.float16
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_ID,
    quantization_config = Quantization_config,
    device_map = "auto"
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
model.config.use_cache = False # Make true during inference

In [ ]:
# Setting tokenizer and chat template
tokenizer = AutoTokenizer.from_pretrained(model_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right" # Right pad for training and left for inference

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
# Prepare dataset

raw = load_dataset("databricks/databricks-dolly-15k", split="train")

def to_chat(example):
    user = example["instruction"]
    if example["context"]:
        user += "\n\n" + example["context"]

    messages = [
        {"role": "user", "content": user},
        {"role": "assistant", "content": example["response"]},
    ]
    # tokenize=False returns the formatted string; SFTTrainer tokenizes it later.
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

dataset = raw.map(to_chat, remove_columns=raw.column_names)

# Inspect one example. Confirm the special tokens look right before training.
print(dataset[0]["text"])

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
When did Virgin Australia start operating?

Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.<|im_end|>
<|im_start|>assistant
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.<|im_end|>



In [ ]:
# Preparing and formating the dataset
from transformers import AutoTokenizer
raw = load_dataset("databricks/databricks-dolly-15k", split = "train")

def format_chat(chat) :
    user_msg = chat["instruction"]
    if chat["context"] :
      user_msg += "\n\n" + chat["context"]

    messages = [
        {"role" : "user", "content" : user_msg},
        {"role" : "assistant", "content" : chat["response"]},
    ]

    text = tokenizer.apply_chat_template(messages, tokenize = False)  # Tokenize = false returns the formatted string, later tokenized by SFTTrainer
    return {"text" : text}

dataset = raw.map(format_chat, remove_columns = raw.column_names)

print(dataset[0])

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

{'text': "<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhen did Virgin Australia start operating?\n\nVirgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.<|im_end|>\n<|im_start|>assistant\nVirgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.<|im_end|>\n"}


In [ ]:
print(dataset[0]["text"])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
When did Virgin Australia start operating?

Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.<|im_end|>
<|im_start|>assistant
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.<|im_end|>



In [ ]:
# Alternative datasets and formatting is not required as they are already formatted through HuggingFace

from datasets import load_dataset

# 1. Stanford Alpaca (Often used as a direct alternative to Dolly)

alpaca = load_dataset("tatsu-lab/alpaca", split="train")
print("--- Alpaca Example ---")
print(f"Columns: {alpaca.column_names}")
print(f"Instruction: {alpaca[0]['instruction']}")
print(f"Output: {alpaca[0]['output']}\n")

# 2. Ultrachat 200k (Great for multi-turn conversations)
# Loading a small subset (10k) for inspection to save time/memory

ultrachat = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft[:10%]")
print("--- Ultrachat 200k Example ---")
print(f"Columns: {ultrachat.column_names}")
# Messages are already formatted in a chat structure
print(f"Messages Structure Sample: {ultrachat[0]['messages']}")

README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

--- Alpaca Example ---
Columns: ['instruction', 'input', 'output', 'text']
Instruction: Give three tips for staying healthy.
Output: 1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. 
2. Exercise regularly to keep your body active and strong. 
3. Get enough sleep and maintain a consistent sleep schedule.



...

Generating train_sft split:   0%|          | 0/207865 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

--- Ultrachat 200k Example ---
Columns: ['prompt', 'prompt_id', 'messages']
Messages Structure Sample: [{'content': "These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?\nOn your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the theme's built-in settings!\nYour Collection pages & Featured Collections sections will now display the secondary product image just by hovering over that product image thumbnail.\nDoes this feature apply to all sections of the theme or just specific ones as listed in the text material?", 'role': 'user'}, {'content': 'This feature only applies to Collection pages and Featured Collections sections of the section-based themes listed in the text material.', 'role': 'assistant'}, {'content': 'Can you guide me through the process of enabling the secondary image hover feature on my Coll

In [ ]:
lora_config = LoraConfig(
    r=16,                       # rank: capacity of the adapter (8-64 typical)
    lora_alpha=32,             # scaling; alpha = 2*r is a safe default
    lora_dropout=0.05,         # dropout on the LoRA input; regularizes small datasets
    bias="none",               # do not train bias terms (standard for LoRA)
    task_type="CAUSAL_LM",     # tells PEFT how to wrap the model
    target_modules=[           # which linear layers get an adapter
        "q_proj", "k_proj", "v_proj", "o_proj",     # attention
        "gate_proj", "up_proj", "down_proj",        # MLP
    ],
    # target_modules="all-linear",  # shortcut: adapt every linear layer
)

In [ ]:
# Setting the LORA configuration

lora_config = LoraConfig(
    r = 8, # Rank of LORA matrix (capacity of adapter)
    lora_alpha = 16, # scaling factor (relativeness btw pretrained and new parameters)
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM",
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj", # Attention
        "gate_proj", "up_proj", "down_proj",   # MLP
    ],
)

In [ ]:
# Training configuration and Trainer optimized to prevent Out Of Memory (OOM) on 15GB GPUs
import gc
import torch

# Free up unused VRAM memory before starting
gc.collect()
torch.cuda.empty_cache()

sft_config = SFTConfig(
    output_dir="qwen2.5-7b-dolly-lora",
    num_train_epochs = 1,
    per_device_train_batch_size = 1,      # Reduced from 4 to 1 to drastically lower peak VRAM usage
    gradient_accumulation_steps = 16,     # Increased to 32 to maintain an effective batch size of 32
    gradient_checkpointing = True,        # Recomputes activations during backward pass to save VRAM
    learning_rate = 2e-4,
    max_length=1024,                      # Changed from max_seq_length back to max_length
    packing = False,
    logging_steps=10,
    save_strategy="no",
    fp16 = True,
    bf16 = False,
    optim = "paged_adamw_8bit",
    lr_scheduler_type = "cosine",
)

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=sft_config,
)

Adding EOS to train dataset:   0%|          | 0/15011 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/15011 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/15011 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [ ]:
trainer.model.print_trainable_parameters()

trainable params: 20,185,088 || all params: 7,635,801,600 || trainable%: 0.2643


In [ ]:
trainer.train()

Step,Training Loss


In [ ]:
trainer.save_model("qwen2.5-7b-dolly-lora/final")

## Inference (A): By merging into the base model: Faster deployment

In [ ]:
from peft import PeftModel

base_fp16 = AutoModelForCausalLM.from_pretrained(model_ID, torch_dtype = torch.float16, device_map = "auto")

merged = PeftModel.from_pretrained(base_fp16, "qwen2.5-7b-dolly-lora/final")
merged = merged.merge_and_unload()  # folds B·A into W0
merged.save_pretrained("qwen2.5-7b-dolly-merged")

tokenizer.save_pretrained("qwen2.5-7b-dolly-merged")

## Inference(B): By keeping the adapters separated

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_ID, quantization_config = Quantization_config , device_map = "auto"
)

model = PeftModel.from_pretrained(base_model, "qwen2.5-7b-dolly-lora/final")
model.eval()

tok = AutoTokenizer.from_pretrained(model_ID)
tok.padding_side = "left"
messages = [{"role" : "user", "content" : "What is the meaning of mobile?"}]

inputs = tok.apply_chat_template(
    messages, add_generation_prompt = True, return_tensors = "pt", return_dict = True,
).to(model.device)

input_len = inputs["input_ids"].shape[1]

output = model.generate(**inputs, max_new_tokens = 200, do_sample = True, temperature = 0.7)
print(tok.decode(output[0][input_len:], skip_special_tokens = True))